# ⚠️ ADVERTENCIA DE PRIVACIDAD

Este notebook se ejecuta **fuera de AURA**, en Google Colab (infraestructura de Google Cloud).

- Los datos que cargues serán procesados en servidores de Google.
- Si tu dataset contiene información sensible, personal o confidencial, **no lo subas a Colab**.
- AURA no envía datos automáticamente. Tú decides qué archivo cargar.
- Para datos sensibles, descarga el script `.py` desde AURA y ejecútalo en tu entorno local (Jupyter, VS Code, terminal).


# AURA — Notebook de Limpieza de Datos

**Dataset:** incidentes_semantic_sample.csv
**Fecha de exportación:** 2026-06-19
**Score AURA:** 65/100
**Filas auditadas:** 10 · **Columnas:** 7
**Hallazgos detectados:** 5

---

## Limitaciones

- La auditoría se ejecutó en navegador con preview de 5.000 filas.
- El script fue generado por IA y aprobado por revisión humana (HITL). **No ha sido ejecutado.**
- La simulación de remediación en AURA es determinista (JavaScript), no Python.
- Este notebook permite ejecutar el script real en Python/Pandas en Colab.
- Después de ejecutar, compara el delta de salud real con el delta simulado en AURA.


## Instrucciones

1. Ejecuta las celdas en orden (Ctrl+Enter o Shift+Enter).
2. Cuando se solicite, sube el archivo CSV original (el mismo que cargaste en AURA).
3. El script de limpieza se ejecutará automáticamente.
4. El dataset corregido se descargará como `dataset_corregido.csv`.
5. Revisa el log de ejecución y el delta de salud.
6. Vuelve a AURA para re-auditar el dataset corregido y comparar resultados.


In [ ]:
# Celda 1: Subir archivo CSV
from google.colab import files
import io

print("Selecciona el archivo CSV original (el mismo que cargaste en AURA):")
uploaded = files.upload()

# Obtener el nombre del archivo subido
filename = list(uploaded.keys())[0]
print(f"Archivo cargado: {filename} ({len(uploaded[filename])} bytes)")


In [ ]:
# Celda 2: Leer CSV con Pandas
import pandas as pd

df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f"Filas: {len(df)}")
print(f"Columnas: {list(df.columns)}")
print(f"Tipos:\n{df.dtypes}")
df.head()


In [ ]:
# Celda 3: Script de limpieza aprobado (HITL)
# Este script fue generado por IA y aprobado por revisión humana en AURA.
# NO MODIFICAR sin antes entender qué hace cada línea.

"""
AURA — Script de limpieza controlado v2 (fixture Incidentes Policiales).
Corrige patrones detectados por el motor determinista de AURA sin
destruir evidencia y sin introducir nulos críticos penalizados por runAudit.

Estrategia de remediación trazable:
- No borrar evidencia original (conservar crimeid_original).
- No dejar CrimeId vacío o nulo (usa placeholder no-tóxico).
- Usar placeholder semántico: "CORRUPTED_ID_REQUIRES_SOURCE_REVIEW".
- Marcar filas afectadas con crimeid_corrupted=True.
- No inventar IDs numéricos ficticios.
- Normalizar City y limpiar espacios.
"""

import pandas as pd

CORRUPTED_ID_PLACEHOLDER = "CORRUPTED_ID_REQUIRES_SOURCE_REVIEW"

DISPOSITION_VOCABULARY = {
    "handled advised": "Handled/Advised",
    "not recorded": "Not Recorded",
    "arrest citation": "Arrest/Citation",
    "gone unable to locate": "Gone/Unable to Locate",
}


def _normalize(value):
    if pd.isna(value) or not isinstance(value, str):
        return ""
    return (
        value.strip()
        .lower()
        .replace("/", " ")
        .replace("  ", " ")
    )


def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Limpia el dataset aplicando correcciones trazables:
    - Marca filas donde CrimeId está contaminado con vocabulario de Disposition.
    - Sustituye CrimeId contaminado por placeholder no-nulo no-tóxico.
    - Retiene el CrimeId original como evidencia en columnas auxiliares.
    - Normaliza casing de City.
    - Recorta espacios externos en columnas string.
    """
    df = df.copy()

    df["crimeid_corrupted"] = False
    df["crimeid_original"] = df["CrimeId"].astype(str)
    df["crimeid_correction_note"] = ""

    for idx in df.index:
        crimeid_raw = str(df.at[idx, "CrimeId"])
        crimeid_norm = _normalize(crimeid_raw)

        if crimeid_norm in DISPOSITION_VOCABULARY:
            df.at[idx, "crimeid_corrupted"] = True
            df.at[idx, "crimeid_correction_note"] = (
                f"CrimeId contenía valor de Disposition: "
                f"'{DISPOSITION_VOCABULARY[crimeid_norm]}'. "
                f"Placeholder usado: '{CORRUPTED_ID_PLACEHOLDER}'. "
                f"Original conservado en crimeid_original."
            )
            df.at[idx, "CrimeId"] = CORRUPTED_ID_PLACEHOLDER

    if "City" in df.columns:
        df["City"] = df["City"].astype(str).str.strip().str.lower()

    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].apply(
                lambda x: x.strip() if isinstance(x, str) else x
            )

    return df



In [ ]:
# Celda 4: Ejecutar limpieza y descargar dataset corregido
import hashlib
from datetime import datetime

print("Ejecutando script de limpieza...")
start_time = datetime.now()

# Ejecutar la función de limpieza
df_corregido = clean_dataset(df)

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

# Calcular hash del resultado
csv_output = df_corregido.to_csv(index=False)
output_hash = hashlib.sha256(csv_output.encode()).hexdigest()

# Guardar archivo corregido
output_filename = "dataset_corregido.csv"
df_corregido.to_csv(output_filename, index=False)

print(f"✅ Limpieza completada en {duration:.2f}s")
print(f"Filas procesadas: {len(df_corregido)}")
print(f"Hash del resultado: {output_hash[:16]}...")
print(f"Columnas: Address, AddressType, CallDateTime, City, CrimeId, Disposition, OriginalCrimeTypeName")

# Descargar archivo corregido
from google.colab import files
files.download(output_filename)

# Mostrar primeras filas del resultado
df_corregido.head()


## Checklist post-ejecución

Después de ejecutar este notebook, completa los siguientes pasos para cerrar el ciclo de evidencia:

- [ ] Guarda el dataset corregido (`dataset_corregido.csv`).
- [ ] Guarda el log de ejecución (tiempo, hash, filas procesadas).
- [ ] **Vuelve a AURA** y carga el dataset corregido para re-auditarlo.
- [ ] Compara el delta de salud real (Colab) con el delta simulado (AURA).
- [ ] Si el delta real es menor que el simulado, revisa el script: puede contener operaciones que no mejoran la calidad.
- [ ] Conserva siempre una copia del dataset original sin modificar.
- [ ] Documenta cualquier decisión manual tomada durante la revisión del script.

---

*Notebook generado por AURA v0.5.0 el 2026-06-19. Dataset: incidentes_semantic_sample.csv. Score: 65/100.*

*Evidencia de trazabilidad incluida en metadatos del notebook.*
